# 🛣️ PotholeFix: AI Pothole Detection Model Training
### Ultralytics YOLOv8 / YOLOv11 Training Notebook for Google Colab (Free T4 GPU)

This notebook trains a custom YOLO model to detect road surface potholes and estimate risk classification for the **PotholeFix** platform.

**Workflow Summary:**
1. **Environment Setup:** Check GPU acceleration and install Ultralytics
2. **Dataset Ingestion:** Mount Google Drive, upload a dataset ZIP, or download via Roboflow
3. **Validation:** Verify bounding box annotations and class configuration
4. **Training:** Train `yolov8n.pt` for 30–50 epochs with early stopping (~15 minutes on T4 GPU)
5. **Evaluation:** Review mAP@50 and confusion matrix
6. **Export:** Download `best.pt` and place into `backend/weights/best.pt`

## Step 1: Check GPU Environment
Verify that Google Colab is using a GPU (Runtime → Change runtime type → T4 GPU).

In [ ]:
# Verify GPU availability
!nvidia-smi

import torch
print(f"CUDA Available: {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"Active Device: {torch.cuda.get_device_name(0)}")
else:
    print("WARNING: No GPU detected! Go to Runtime -> Change runtime type -> T4 GPU.")

## Step 2: Install Ultralytics YOLO & Dependencies

In [ ]:
!pip install -q ultralytics pyyaml matplotlib pillow

## Step 3: Dataset Ingestion
Choose **Option A** (Google Drive mount), **Option B** (direct file upload), or **Option C** (Roboflow Universe download).

In [ ]:
# OPTION A: Mount Google Drive (Recommended for larger datasets)
# from google.colab import drive
# drive.mount('/content/drive')
# !unzip -q "/content/drive/MyDrive/pothole_dataset.zip" -d /content/dataset

# OPTION B: Upload a ZIP directly in Colab
import os
from google.colab import files

print("Upload your dataset .zip containing images/ and labels/ directories:")
uploaded = files.upload()
for filename in uploaded.keys():
    if filename.endswith('.zip'):
        os.makedirs('/content/dataset', exist_ok=True)
        !unzip -q "{filename}" -d /content/dataset
        print(f"Extracted {filename} to /content/dataset")

## Step 4: Configure `data.yaml`
Ensure `data.yaml` points to `/content/dataset` with class `0: pothole`.

In [ ]:
import yaml
from pathlib import Path

data_yaml_content = """
path: /content/dataset
train: images/train
val: images/val
test: images/test

names:
  0: pothole
"""

with open('/content/dataset/data.yaml', 'w') as f:
    f.write(data_yaml_content.strip())

print("Created /content/dataset/data.yaml:")
!cat /content/dataset/data.yaml

## Step 5: Verify Dataset Contents
Count images and verify structure.

In [ ]:
import glob

train_imgs = glob.glob('/content/dataset/images/train/*.*')
val_imgs = glob.glob('/content/dataset/images/val/*.*')
train_lbls = glob.glob('/content/dataset/labels/train/*.txt')

print(f"Training images:   {len(train_imgs)}")
print(f"Training labels:   {len(train_lbls)}")
print(f"Validation images: {len(val_imgs)}")

## Step 6: Train YOLO Object Detector
Train `yolov8n.pt` with 50 epochs, batch 16, image resolution 640.

In [ ]:
from ultralytics import YOLO

# Load lightweight pre-trained model
model = YOLO('yolov8n.pt')

# Start training
results = model.train(
    data='/content/dataset/data.yaml',
    epochs=50,
    batch=16,
    imgsz=640,
    device=0,
    patience=15,
    project='potholefix_runs',
    name='yolov8_pothole',
    save=True,
    plots=True,
)

## Step 7: Review Training Results & Metrics

In [ ]:
from IPython.display import Image, display

# Display training curves and confusion matrix
run_dir = '/content/potholefix_runs/yolov8_pothole'
for plot_name in ['results.png', 'confusion_matrix.png']:
    plot_path = f"{run_dir}/{plot_name}"
    if os.path.exists(plot_path):
        print(f"=== {plot_name} ===")
        display(Image(plot_path, width=800))

## Step 8: Validate on Test Pothole Image

In [ ]:
# Run inference with best weights
trained_model = YOLO('/content/potholefix_runs/yolov8_pothole/weights/best.pt')

# Predict on sample validation images
test_results = trained_model.predict(
    source='/content/dataset/images/val',
    conf=0.35,
    save=True,
    max_det=5,
)

print(f"Inference complete on {len(test_results)} validation images.")

## Step 9: Download `best.pt` Model Weights
Download `best.pt` and place it into `backend/weights/best.pt` on your local machine.

In [ ]:
from google.colab import files
best_weights = '/content/potholefix_runs/yolov8_pothole/weights/best.pt'

if os.path.exists(best_weights):
    print(f"Downloading {best_weights}...")
    files.download(best_weights)
    print("SUCCESS! Copy this file to your project: potholefix/backend/weights/best.pt")
else:
    print("ERROR: best.pt weights file not found.")